# PrimeVarClass — Pontuação ESM-2 (GPU)

Este notebook pontua as variantes missense de BRCA1/BRCA2 com o **ESM-2 (650M)** de
Lin et al. (Science, 2023), usando LLR *masked-marginal* em janela.

**Como usar:**
1. `Ambiente de execução` → `Alterar o tipo de ambiente de execução` → **GPU (T4)**.
2. `Ambiente de execução` → `Executar tudo`.
3. Ao final, o arquivo **`esm2_scores.csv`** será baixado automaticamente.
4. Salve esse arquivo em `scratch/esm_input/esm2_scores.csv` no projeto e avise o assistente.

Tempo esperado na T4: ~5–10 minutos.


In [ ]:
# 1) Instalar dependências e checar GPU
!pip -q install "transformers>=4.40" "torch" 2>/dev/null
import torch

print("GPU disponível:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# 2) Baixar insumos via git clone (robusto contra rate-limit do raw CDN)
!git clone -q https://github.com/WesleyCapucho/primevarclass /content/pvc
import pandas as pd

D = "/content/pvc/scratch/esm_input/"
SEQ = {"BRCA1": open(D + "BRCA1_P38398.txt").read().strip(),
       "BRCA2": open(D + "BRCA2_P51587.txt").read().strip()}
var = pd.read_csv(D + "brca_variants_unique.csv")
print("BRCA1:", len(SEQ["BRCA1"]), "aa | BRCA2:", len(SEQ["BRCA2"]), "aa | variantes:", len(var))


In [ ]:
# 3) Carregar ESM-2 650M
from transformers import AutoModelForMaskedLM, AutoTokenizer

MODEL = "facebook/esm2_t33_650M_UR50D"
dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForMaskedLM.from_pretrained(MODEL).to(dev).eval()
AAS = list("ACDEFGHIKLMNPQRSTVWY")
aa_ids = {a: tok.convert_tokens_to_ids(a) for a in AAS}
mask_id = tok.mask_token_id
print("modelo carregado em", dev)


In [ ]:
# 4) Pontuar (masked-marginal em janela; uma passagem por posição -> todas as substituições)
import time

W = 511
pos_df = var[["gene", "position", "aa_ref"]].drop_duplicates().reset_index(drop=True)
cache = {}
t0 = time.time(); skipped = 0
for i, r in pos_df.iterrows():
    g, p, ref = str(r["gene"]), int(r["position"]), str(r["aa_ref"])
    seq = SEQ[g]
    if p < 1 or p > len(seq) or seq[p-1] != ref:
        skipped += 1; continue
    s = max(0, p-1-W); e = min(len(seq), p-1+W+1); win = seq[s:e]; loc = (p-1)-s
    enc = tok(win, return_tensors="pt").to(dev); ids = enc["input_ids"]; tp = loc + 1
    ids[0, tp] = mask_id
    with torch.no_grad():
        lp = torch.log_softmax(model(ids).logits[0, tp], dim=-1)
    cache[(g, p)] = {a: float(lp[aa_ids[a]]) for a in AAS}
    if (i+1) % 200 == 0:
        print(f"{i+1}/{len(pos_df)}  ({time.time()-t0:.0f}s)")
print("posições pontuadas:", len(cache), "| puladas (mismatch):", skipped, f"| {time.time()-t0:.0f}s")


In [ ]:
# 5) Montar LLR por variante e baixar
rows = []
for _, r in var.iterrows():
    g, p, ref, alt = str(r["gene"]), int(r["position"]), str(r["aa_ref"]), str(r["aa_alt"])
    lp = cache.get((g, p))
    if lp and ref in lp and alt in lp:
        rows.append({"gene": g, "position": p, "aa_ref": ref, "aa_alt": alt,
                     "esm2_llr": round(lp[alt] - lp[ref], 5)})
out = pd.DataFrame(rows)
out.to_csv("esm2_scores.csv", index=False)
print("variantes pontuadas:", len(out), "| LLR mean", round(out.esm2_llr.mean(), 3))
# sanity: pathogenic RING cysteines should be strongly negative
print(out[(out.gene=="BRCA1") & (out.position.isin([61, 64]))])
from google.colab import files

files.download("esm2_scores.csv")
